# Do stock prices and revenues tell the same story?
### Tesla vs. GameStop · a reproducible historical case study

An extension of the IBM / Coursera *Extracting and Visualizing Stock Data* project. This version turns the extraction exercise into a comparative analysis with validated data, explicit assumptions, interactive charts, and reusable Python functions.

**Research question:** How did adjusted stock performance, downside risk, and quarterly revenue growth differ between these two businesses over a shared historical window?

**Scope:** Historical exploration, not a valuation model or investment recommendation. Revenue is one business measure; it does not explain stock prices on its own.

## 1. Reproduce the analysis

Use Python 3.11+ and install `requirements.txt` in a virtual environment (see README). Run this notebook from the project directory. All analytical functions live in `analysis.py`, which is also usable from the command line and covered by offline tests.

The included snapshots allow a network-free rerun. Set `REFRESH = True` only when you intentionally want to download replacement data. The IBM revenue sources are archived pages; refreshing them does **not** produce current financial statements.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

from analysis import ROOT, COMPANIES, parse_revenue, revenue_features, run

START_DATE = "2010-07-01"
END_DATE = None  # Automatically use the older of the two final revenue dates.
REFRESH = False
OUTPUT_DIR = ROOT / "reports"

## 2. Extract and validate quarterly revenue

Instead of selecting the second HTML table by position, locate the company's **quarterly revenue** heading. Parse dates and monetary values once, remove blank revenue rows, reject malformed values and duplicate dates, then sort chronologically. Dollar values are reported in **USD millions**, as labeled in the source.

The parser and retrieval functions are separate so parsing can be tested without network access. Downloads use HTTP status checks, timeouts, and bounded retries.

In [2]:
revenues = {}
coverage = []
for ticker, (company, source_url) in COMPANIES.items():
    source = ROOT / "data" / "raw" / f"{ticker}_revenue.html"
    frame = parse_revenue(source.read_text(encoding="utf-8"), company)
    revenues[ticker] = revenue_features(frame)
    coverage.append({
        "Ticker": ticker,
        "First fiscal date": frame["Date"].min().date(),
        "Last fiscal date": frame["Date"].max().date(),
        "Quarters": len(frame),
        "Source": source_url,
    })
display(pd.DataFrame(coverage).set_index("Ticker"))
for ticker, frame in revenues.items():
    print(f"{ticker} · latest observations · revenue in USD millions")
    display(frame.tail(5).round(2))

,First fiscal date,Last fiscal date,Quarters,Source
Ticker,,,,
TSLA,2009-06-30,2022-09-30,53,https://cf-courses-data.s3.us.cloud-object-sto...
GME,2005-01-31,2020-04-30,62,https://cf-courses-data.s3.us.cloud-object-sto...


TSLA · latest observations · revenue in USD millions


/var/folders/44/967rz3n1533gknzgln8pg3100000gn/T/ipykernel_42590/2021910235.py:17: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(frame.tail(5).round(2))


,Date,Revenue,Revenue_YoY_pct,Revenue_TTM
48,2021-09-30,13757.0,56.85,46848.0
49,2021-12-31,17719.0,64.92,53823.0
50,2022-03-31,18756.0,80.54,62190.0
51,2022-06-30,16934.0,41.61,67166.0
52,2022-09-30,21454.0,55.95,74863.0


GME · latest observations · revenue in USD millions


/var/folders/44/967rz3n1533gknzgln8pg3100000gn/T/ipykernel_42590/2021910235.py:17: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(frame.tail(5).round(2))


,Date,Revenue,Revenue_YoY_pct,Revenue_TTM
57,2019-04-30,1548.0,-13.33,8047.0
58,2019-07-31,1286.0,-14.32,7832.0
59,2019-10-31,1439.0,-25.63,7336.0
60,2020-01-31,2194.0,-28.37,6467.0
61,2020-04-30,1021.0,-34.04,5940.0


## 3. Define a fair comparison

Use the intersection of observed trading dates, with each adjusted closing-price series rebased to **100 on the same date**. An indexed series makes relative performance comparable despite different share prices and stock splits.

Yahoo Finance prices are fetched with `auto_adjust=True`. These are adjusted historical prices, which may be revised by the provider and are not the nominal share prices quoted on those historical days. Do not adjust them a second time.

The default end date is the older of the final revenue dates. This deliberately excludes the 2021 GameStop episode: the supplied GameStop revenue archive cannot support a matched comparison for that period.

In [3]:
result = run(output=OUTPUT_DIR, refresh=REFRESH, start=START_DATE, end=END_DATE)
prices = result["prices"]
metrics = result["metrics"]
manifest = result["manifest"]
print(f"Comparison: {manifest['comparison_start']} to {manifest['comparison_end']}")
print(f"Common trading dates: {len(prices):,}")
display(prices.head().round(3))

Comparison: 2010-07-01 to 2020-04-30
Common trading dates: 2,475


,TSLA,GME
Date,,
2010-07-01,1.464,3.210
2010-07-02,1.280,3.075
2010-07-06,1.074,3.097
2010-07-07,1.053,3.156
2010-07-08,1.164,3.173


## 4. Compare performance and risk

- **Cumulative return:** `(last adjusted close / first adjusted close − 1) × 100`.
- **Annualized volatility:** sample standard deviation of daily simple returns × `√252` × 100; 252 is a trading-day convention.
- **Maximum drawdown:** the lowest `(price / running maximum − 1) × 100` within the selected window. More negative values indicate deeper peak-to-trough declines.
- **Revenue YoY growth:** change against the same fiscal quarter one year earlier. Missing quarters remain missing; zero denominators yield no growth estimate.
- **Trailing twelve-month revenue:** sum of four consecutive quarters; unavailable when any quarter is absent.

The two companies have different fiscal calendars. The code maps observations to calendar-quarter bins to identify year-earlier fiscal observations; it rejects multiple observations per bin. An issuer with a changing fiscal calendar would require a dedicated fiscal-period mapping.

In [4]:
display(metrics.style.format("{:.2f}%"))
for ticker, row in metrics.iterrows():
    print(
        f"{ticker}: adjusted-price change {row['Return_pct']:+,.1f}%; "
        f"annualized volatility {row['Annualized_volatility_pct']:.1f}%; "
        f"maximum drawdown {row['Max_drawdown_pct']:.1f}%."
    )

,Return_pct,Annualized_volatility_pct,Max_drawdown_pct
Ticker,,,
TSLA,3460.47%,54.65%,-60.63%
GME,-55.37%,47.53%,-93.21%


TSLA: adjusted-price change +3,460.5%; annualized volatility 54.7%; maximum drawdown -60.6%.
GME: adjusted-price change -55.4%; annualized volatility 47.5%; maximum drawdown -93.2%.


## 5. Interactive dashboard

Hover for values, click a legend entry to toggle that company's traces, and drag to zoom. The exported `reports/dashboard.html` embeds Plotly so it can open offline. Revenue panels are clipped to the same window as the price panels, while growth calculations can use earlier revenue observations.

Separate panels preserve honest units rather than overlaying price and revenue on independently scaled axes.

In [5]:
figure = result["figure"]
figure.show(renderer="plotly_mimetype")

## 6. Interpret the evidence

For the included **July 2010–April 2020** window, Tesla's adjusted-price series rises sharply while GameStop's declines. Both experience substantial drawdowns. This illustrates why comparing headline returns alone hides the risk borne along the way.

Read the revenue plots alongside those outcomes: they provide business context, but revenue is not profit, free cash flow, or valuation. These companies also differ in industry, maturity, capital structure, and exposure to market conditions. A two-stock sample cannot establish a general relationship.

**Why no price/revenue correlation score?** Daily prices and quarterly revenues have different frequencies. Trending levels can produce misleading correlations, and fiscal period-end dates precede public earnings announcements. A defensible event study needs verified publication timestamps and a benchmark. This project therefore makes descriptive comparisons without presenting correlation as causation or a trading signal.

**Next research step:** collect filing publication dates and a broad market benchmark, then test benchmark-adjusted returns around earnings announcements using only information available at each point in time.

## 7. Audit the run

The manifest records the actual date window, sample size, URLs, and SHA-256 hashes of input snapshots. It distinguishes the report generation time from the historical observation dates. CSV exports preserve the cleaned revenue, aligned prices, and calculated metrics.

In [6]:
display(pd.DataFrame(manifest["sources"]).T)
print("Saved artifacts:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(path.name)

,revenue_url,revenue_sha256,revenue_last_date,prices_sha256,price_source
TSLA,https://cf-courses-data.s3.us.cloud-object-sto...,dcaeb19abdc832f5b9250241c3352053ecd98894b6ce57...,2022-09-30,6cbfda1bedc32eba0b857850fb7b3c662dbe0c2412e65f...,Yahoo Finance via yfinance; auto_adjust=True
GME,https://cf-courses-data.s3.us.cloud-object-sto...,73154deae9bb0d98d3e2af54f149af379bf578e6c5d540...,2020-04-30,8d74660ac81c1579f8d4476c9610b32230041e517664b8...,Yahoo Finance via yfinance; auto_adjust=True


Saved artifacts:
GME_revenue.csv
TSLA_revenue.csv
aligned_prices.csv
dashboard.html
manifest.json
metrics.csv


## Sources and attribution

- Original learning exercise: IBM / Coursera, *Extracting and Visualizing Stock Data*, supplied in the user's notebook. The original course notebook remains unchanged.
- Course notebook attribution: Joseph Santarcangelo and Azim Hirjani; original notice: **Copyright © 2020 IBM Corporation. All rights reserved.**
- Revenue: IBM-hosted historical Macrotrends page snapshots for [Tesla](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm) and [GameStop](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html).
- Prices: Yahoo Finance through [yfinance](https://ranaroussi.github.io/yfinance/). Its data access and third-party data remain subject to their respective terms.

Engineering and analytical extensions include reusable extraction, validation, snapshot-based reproducibility, shared-window comparisons, risk metrics, revenue features, automated tests, and an interactive report. Third-party content and data retain their existing rights; no new license is granted for them.